### D. Generate Descriptions

In [ ]:
import torch

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

def generate_description(input_text, max_length=256, top_p=0.9, temperature=0.7, use_greedy=False):
    input_text = f'{input_text}<SEP>'
    inputs = tokenizer(input_text, return_tensors='pt', max_length=128, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    if use_greedy:
        # Generate with greedy search
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_length,
            no_repeat_ngram_size=2,
            early_stopping=True
        )
    else:
        # Generate with top-p sampling
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_length,
            top_p=top_p,
            temperature=temperature,
            no_repeat_ngram_size=2,
            do_sample=True,
            early_stopping=True
        )
    
    description = tokenizer.decode(outputs[0], skip_special_tokens=True)
    description = description.split('<SEP>')[-1].strip().capitalize()
    return description

# Example with greedy search
input_text = 'Style: Industrial | Material: Metal | Color: Gray | Dimensions: 30L x 20W x 15H'
print('Greedy Search:')
print(generate_description(input_text, use_greedy=True))
print('\nTop-p Sampling:')
print(generate_description(input_text, use_greedy=False))

In [ ]:
# Load fine-tuned model and tokenizer
model = GPT2LMHeadModel.from_pretrained(f'{base_path}/fine-tuned-gpt2')
tokenizer = GPT2Tokenizer.from_pretrained(f'{base_path}/fine-tuned-gpt2')
tokenizer.pad_token = tokenizer.eos_token

def generate_product_description(input_features, max_length=256, top_p=0.9, temperature=0.7, use_greedy=False):
    """
    Generate product description from input features

    Args:
        input_features (str): Product features in format 'Style:...|Material:...|Color:...'
        max_length (int): Maximum length of generated description
        top_p (float): Top-p sampling parameter (ignored if use_greedy=True)
        temperature (float): Temperature for sampling (ignored if use_greedy=True)
        use_greedy (bool): If True, use greedy search; otherwise, use top-p sampling

    Returns:
        str: Generated product description
    """
    input_text = f'{input_features}<SEP>'
    inputs = tokenizer(
        input_text,
        max_length=128,
        truncation=True,
        return_tensors='pt'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        if use_greedy:
            outputs = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_length=max_length,
                no_repeat_ngram_size=2,
                early_stopping=True
            )
        else:
            outputs = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_length=max_length,
                top_p=top_p,
                temperature=temperature,
                no_repeat_ngram_size=2,
                do_sample=True,
                early_stopping=True
            )

    description = tokenizer.decode(outputs[0], skip_special_tokens=True)
    description = description.split('<SEP>')[-1].strip().capitalize()

    return description

# Example usage
input_features = 'Style: Industrial | Material: Metal | Color: Gray | Dimensions: 30L x 20W x 15H | Features: Rust-proof, Wall-mounted'
print('Input Features:')
print(input_features)
print('\nGreedy Search Description:')
print(generate_product_description(input_features, use_greedy=True))
print('\nTop-p Sampling Description:')
print(generate_product_description(input_features, use_greedy=False))